# Anemia (registro administrativo)

## SIEN / REUNIS (MINSA)

Fuente: https://www.minsa.gob.pe/reunis/

No tiene descarga directa en CSV — requiere revisar el tablero o solicitar la data en otro formato.

In [7]:
from pathlib import Path
import pandas as pd

def find_project_root(marker="requirements.txt"):
    path = Path.cwd()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"No se encontró '{marker}' subiendo desde {path}")

PROJECT_ROOT = find_project_root()
DATA = PROJECT_ROOT / "data"

ruta_sien = DATA / "raw" / "Minsa_reunis_anemia" / "Trama_Base_Anemia.xlsx"
sien = pd.read_excel(ruta_sien, dtype={"Ubigeo": str})

print(sien.shape)
sien.head()

(355876, 15)


,Año,Edad,Diresa,Departamento,Provincia,Distrito,Renipres,Ubigeo,Sexo,Evaluados,Anemia,Anemia Leve,Anemia Moderada,Anemia Severa,Normal
0,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5053.0,010202,M,12,0,0,0,0,12
1,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5053.0,010202,F,14,0,0,0,0,14
2,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5054.0,010202,M,3,0,0,0,0,3
3,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5054.0,010202,F,4,0,0,0,0,4
4,2020,1,AMAZONAS,AMAZONAS,BAGUA,ARAMANGO,5055.0,010202,M,6,0,0,0,0,6


In [9]:
# Filtrar solo menores de 3 años (Edad == 1), según confirmamos contra el tablero
sien_menores_3 = sien[sien["Edad"] == 1].copy()

anemia_distrital = (
    sien_menores_3
    .groupby(["Ubigeo", "Año"], as_index=False)
    .agg(
        departamento=("Departamento", "first"),
        provincia=("Provincia", "first"),
        distrito=("Distrito", "first"),
        ninos_evaluados=("Evaluados", "sum"),
        ninos_con_anemia=("Anemia", "sum"),
        ninos_sin_anemia=("Normal", "sum"),
    )
    .rename(columns={"Ubigeo": "ubigeo"})
)

anemia_distrital["prevalencia_anemia"] = (
    anemia_distrital["ninos_con_anemia"] / anemia_distrital["ninos_evaluados"]
)

print(anemia_distrital.shape)
anemia_distrital.head()

(13029, 9)


,ubigeo,Año,departamento,provincia,distrito,ninos_evaluados,ninos_con_anemia,ninos_sin_anemia,prevalencia_anemia
0,010101,2020,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,245,85,160,0.346939
1,010101,2021,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,234,71,163,0.303419
2,010101,2022,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,201,76,125,0.378109
3,010101,2023,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,837,160,677,0.191159
4,010101,2024,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,969,157,812,0.162023


In [10]:
anio_min = anemia_distrital["Año"].min()
anio_max = anemia_distrital["Año"].max()

output_path = DATA / "clean" / "staging" / f"sien_reunis_distrital_{anio_min}_{anio_max}.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

anemia_distrital.to_csv(output_path, index=False)

print(f"Guardado: {output_path}")
print(anemia_distrital.shape)

Guardado: c:\Users\JHOSSEP\Documents\REPO\causal-anemia-model\data\clean\staging\sien_reunis_distrital_2020_2026.csv
(13029, 9)
